# ESM3Di: Predict 3Di Structural Sequences from Amino Acids

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/ESM3di/blob/main/ESM3Di_Colab.ipynb)

This notebook predicts **3Di structural sequences** from amino acid sequences using ESM models fine-tuned with LoRA adapters.

**3Di** is a structural alphabet used by [FoldSeek](https://github.com/steineggerlab/foldseek) for fast structure search. Each 3Di character encodes local structural features, enabling sequence-based structure comparison.

## Quick Start
1. Run the **Setup** cell to install dependencies
2. Run the **Download Model** cell to get the pre-trained checkpoint
3. Paste your sequences or upload a FASTA file
4. Run inference and download results

**GPU recommended** - Go to Runtime → Change runtime type → T4 GPU

## 1. Setup

Install ESM3Di and its dependencies. This takes ~2-3 minutes on first run.

In [ ]:
# Install ESM3Di from GitHub
!pip install -q git+https://github.com/YOUR_USERNAME/ESM3di.git

# Additional dependencies
!pip install -q transformers peft torch biopython huggingface_hub

# Verify installation
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ No GPU detected. Inference will be slow. Go to Runtime → Change runtime type → T4 GPU")

## 2. Download Pre-trained Model

Download the ESM3Di checkpoint from HuggingFace Hub. Choose between:
- **ESM2 (35M)** - Smaller (~131MB), faster, well-tested
- **ESM++ (333M)** - Larger (~1.4GB), trained on BFVD dataset, better accuracy

In [ ]:
#@title Select Model { display-mode: "form" }
#@markdown Choose which pre-trained model to use:

model_choice = "ESM2_35M" #@param ["ESM2_35M", "ESMplusplus_BFVD"]

import os
from pathlib import Path
from huggingface_hub import hf_hub_download

# HuggingFace repository containing checkpoints
HF_REPO_ID = "YOUR_USERNAME/esm3di-checkpoints"

# Create checkpoints directory
os.makedirs("checkpoints", exist_ok=True)

# Model configurations
MODEL_CONFIGS = {
    "ESM2_35M": {
        "hf_filename": "esm2_35m_epoch_3.pt",
        "local_path": "checkpoints/esm2_35m_epoch_3.pt",
        "hf_model_name": "facebook/esm2_t12_35M_UR50D",
        "description": "ESM2 35M parameters (~131MB)"
    },
    "ESMplusplus_BFVD": {
        "hf_filename": "esmpp_bfvd_epoch_3.pt",
        "local_path": "checkpoints/esmpp_bfvd_epoch_3.pt",
        "hf_model_name": "Synthyra/ESMplusplus_small",
        "description": "ESM++ 333M trained on BFVD (~1.4GB)"
    }
}

config = MODEL_CONFIGS[model_choice]
checkpoint_path = config["local_path"]
hf_model_name = config["hf_model_name"]

# Download from HuggingFace Hub if not exists
if not Path(checkpoint_path).exists():
    print(f"Downloading {model_choice} checkpoint from HuggingFace Hub...")
    print(f"  Repository: {HF_REPO_ID}")
    print(f"  File: {config['hf_filename']}")
    print(f"  Size: {config['description']}")
    print()
    
    downloaded_path = hf_hub_download(
        repo_id=HF_REPO_ID,
        filename=config["hf_filename"],
        local_dir="checkpoints",
        local_dir_use_symlinks=False
    )
    
    # Rename if needed
    if downloaded_path != checkpoint_path:
        os.rename(downloaded_path, checkpoint_path)
    
    print(f"✓ Downloaded to {checkpoint_path}")
else:
    print(f"✓ Checkpoint already exists: {checkpoint_path}")

print(f"\nSelected model: {model_choice}")
print(f"Base model: {hf_model_name}")
print(f"Checkpoint: {checkpoint_path}")

## 3. Input Sequences

Choose one of the following input methods:
- **Option A**: Paste sequences directly
- **Option B**: Upload a FASTA file

In [ ]:
#@title Option A: Paste Sequences { display-mode: "form" }
#@markdown Enter one or more protein sequences in FASTA format:

fasta_text = """>example_protein_1
MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLSGAEKAVQVKVKALPDAQFEVVHSLAKWKRQQIA
>example_protein_2
KALTARQQEVFDLIRDHISQTGMPPTRAEIAQRLGFRSPNAAEEHLKALARKGVIEIVSGASRGIRLLQEE
""" #@param {type:"string"}

# Save to temporary file
input_fasta = "input_sequences.fasta"
with open(input_fasta, "w") as f:
    f.write(fasta_text)

# Count sequences
num_seqs = fasta_text.count(">")
print(f"✓ Saved {num_seqs} sequence(s) to {input_fasta}")

In [ ]:
#@title Option B: Upload FASTA File { display-mode: "form" }
#@markdown Run this cell to upload a FASTA file from your computer.

from google.colab import files

print("Select a FASTA file to upload...")
uploaded = files.upload()

if uploaded:
    input_fasta = list(uploaded.keys())[0]
    print(f"\n✓ Uploaded: {input_fasta}")
    
    # Count sequences
    with open(input_fasta, 'r') as f:
        num_seqs = sum(1 for line in f if line.startswith('>'))
    print(f"  Contains {num_seqs} sequence(s)")
else:
    print("No file uploaded. Using sequences from Option A.")

## 4. Run Inference

Predict 3Di sequences from the input amino acid sequences.

In [ ]:
import torch
from ESM3di_model import ESM3DiModel, read_fasta

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load checkpoint to get configuration
print(f"\nLoading checkpoint: {checkpoint_path}")
checkpoint = torch.load(checkpoint_path, map_location=device)

# Extract model configuration
args_dict = checkpoint.get('args', {})
num_labels = len(checkpoint.get('label_vocab', []))
use_cnn_head = args_dict.get('use_cnn_head', False)
lora_r = args_dict.get('lora_r', 8)
lora_alpha = args_dict.get('lora_alpha', 16)
lora_dropout = args_dict.get('lora_dropout', 0.05)

print(f"  Labels: {num_labels}")
print(f"  CNN Head: {use_cnn_head}")
print(f"  LoRA r={lora_r}, alpha={lora_alpha}")

# Initialize model
print(f"\nInitializing {hf_model_name}...")
model = ESM3DiModel(
    hf_model_name=hf_model_name,
    num_labels=num_labels,
    lora_r=lora_r,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    use_cnn_head=use_cnn_head,
    cnn_num_layers=args_dict.get('cnn_num_layers', 2),
    cnn_kernel_size=args_dict.get('cnn_kernel_size', 3),
    cnn_dropout=args_dict.get('cnn_dropout', 0.1)
)

# Output file
output_fasta = "predicted_3di.fasta"

# Run inference
print(f"\nRunning inference on {input_fasta}...")
model.predict_from_fasta(
    input_fasta_path=input_fasta,
    output_fasta_path=output_fasta,
    model_checkpoint_path=checkpoint_path,
    batch_size=4,
    device=device
)

print(f"\n✓ Predictions saved to: {output_fasta}")

## 5. View Results

Display the predicted 3Di sequences alongside the input amino acid sequences.

In [ ]:
from ESM3di_model import read_fasta

# Read input and output
aa_records = read_fasta(input_fasta)
three_di_records = read_fasta(output_fasta)

print(f"{'='*80}")
print(f"PREDICTIONS ({len(three_di_records)} sequences)")
print(f"{'='*80}\n")

for (aa_header, aa_seq), (di_header, di_seq) in zip(aa_records, three_di_records):
    print(f">{aa_header}")
    print(f"AA:  {aa_seq[:60]}{'...' if len(aa_seq) > 60 else ''}")
    print(f"3Di: {di_seq[:60]}{'...' if len(di_seq) > 60 else ''}")
    print(f"Length: {len(aa_seq)} residues\n")

In [ ]:
#@title Show Full Output { display-mode: "form" }
#@markdown View the complete 3Di FASTA file:

print("Full 3Di FASTA output:\n")
with open(output_fasta, 'r') as f:
    print(f.read())

## 6. Download Results

Download the predicted 3Di sequences as a FASTA file.

In [ ]:
from google.colab import files

print(f"Downloading {output_fasta}...")
files.download(output_fasta)
print("✓ Download started")

## 7. Create FoldSeek Database (Optional)

If you want to search against other structures, create a FoldSeek database from your predictions.

In [ ]:
#@title Create FoldSeek Database { display-mode: "form" }
#@markdown Install FoldSeek and create a searchable database:

# Install FoldSeek
!wget -q https://mmseqs.com/foldseek/foldseek-linux-avx2.tar.gz
!tar -xzf foldseek-linux-avx2.tar.gz
!rm foldseek-linux-avx2.tar.gz

# Add to PATH
import os
os.environ['PATH'] = os.getcwd() + '/foldseek/bin:' + os.environ['PATH']

# Verify installation
!foldseek version

print("\n" + "="*60)
print("Creating FoldSeek database...")
print("="*60)

# Create database using the fastas2foldseekdb module
from fastas2foldseekdb import create_foldseek_db_from_fastas

db_name = "my_foldseek_db"
success = create_foldseek_db_from_fastas(
    aa_fasta=input_fasta,
    three_di_fasta=output_fasta,
    output_db=db_name
)

if success:
    print(f"\n✓ FoldSeek database created: {db_name}")
    print(f"\nTo search against this database:")
    print(f"  foldseek easy-search query.pdb {db_name} result.m8 tmp")
else:
    print("\n✗ Failed to create database")

---

## About ESM3Di

ESM3Di uses ESM-2 or ESM++ protein language models fine-tuned with LoRA adapters to predict 3Di structural sequences from amino acid sequences alone.

**3Di alphabet** encodes local structural features using 20 characters (similar to amino acids), enabling fast structure comparison without explicit 3D coordinates.

**Resources:**
- [ESM3Di GitHub Repository](https://github.com/YOUR_USERNAME/ESM3di)
- [FoldSeek Paper](https://www.nature.com/articles/s41587-023-01773-0)
- [ESM-2 Paper](https://www.science.org/doi/10.1126/science.ade2574)

**Citation:**
If you use ESM3Di, please cite:
- ESM-2: Lin et al., 2022
- 3Di: van Kempen et al., 2023